### "Statistics and Data Analysis" 31: Density Estimation with Gaussion Mixture Models and Empirical Priors
In this lecture, Dr. Brunton describes how to estimate more complex distributions using empirical distributions given by Gaussian mixture models (GMM).  This is a baby step towards machine learning. 

https://www.youtube.com/watch?v=a1pvm1QGXYg&list=PLMrJAkhIeNNT14qn1c5qdL29A1UaHamjx&index=31

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity

# Function to compute the likelihood of the data given theta
def likelihood(theta, data):
    # Binomial likelihood for coin flips (Binomial trials, p=theta)
    heads = np.sum(data=='H')
    tails = np.sum(data=='T')
    return theta**heads * (1-theta)**tails

# Function to update the posterior distribution using KDE
def bayesian_update_kde(prior_theta, data):
    # Compute the empirical likelihood of the data
    likelihoods = likelihood(prior_theta, data)

    # Weight the prior sample by the likelihood
    posterior_weights = likelihoods / np.sum(likelihoods)

    # Resample from the prior based on the likelihoods to generate posterior theta
    posterior_theta = np.random.choice(prior_theta, size=1000, p=posterior_weights)

    # Use Kernel Density Estimation to estimate the posterior distribution
    kde = KernelDensity(kernel='gaussian', bandwidth=0.05).fit(posterior_theta[:, np.newaxis])
    
    return posterior_theta, kde

# Start with a Prior: Uniform distribution (between 0 and 1)
prior_theta = np.random.uniform(0, 1, size=1000)

# Simulate a sequence of coin flips (H for heads, T for tails)
coin_flips = np.array(['H','T','H','H','T','H','T','T'])

# Store prior values for plotting
theta_list = [prior_theta]

# Perform Bayesian updates after each coin flip
for i, flip in enumerate(coin_flips):
    # print(f"Before Bayesian update: prior_theta={prior_theta}, flip={flip}")
    
    # Update the posterior using the KDE method with the new data (one flip)
    prior_theta, kde = bayesian_update_kde(prior_theta, np.array([flip]))
    
    # Store the updated posterior theta
    theta_list.append(prior_theta)

    # Print the mean of the updated posterior theta after each coin flip
    print(f"After observing {i+1} flips ({flip}): Posterior mean={np.mean(prior_theta):.4f}")

# Plot the evolution of the posterior distribution after all flips
theta_range = np.linspace(0, 1, 100)
plt.figure(figsize=(10, 6))

cmap = plt.get_cmap('viridis_r')
colors = cmap(np.linspace(0, 1, len(theta_list)))

for i, thetas in enumerate(theta_list):
    # Estimate the PDF using the KDE
    kde = KernelDensity(kernel='gaussian', bandwidth=0.05).fit(thetas[:, np.newaxis])
    log_density = kde.score_samples(theta_range[:, np.newaxis])
    density = np.exp(log_density)
    
    # Plot the posterior distibution
    plt.plot(theta_range, density, color=colors[i], label=f"After {i} flips")

# Customize the plot
plt.title("Evolution of Posterior Distribution with Each Coin Flip (KDE)")
plt.xlabel("Theta (probability of heads)")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# Generate 100 random coin flips (1 for heads, 0 for tails)
coin_flips = np.random.choice([1, 0], size=100)

# Start with a prior: Uniform distribution (between 0 and 1)
prior_theta = np.random.uniform(0, 1, size=1000)

# Set up the figure with two subplots: one for the density and one for the mean
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

# Adjust the space between the subplots
plt.subplots_adjust(hspace=0.3)

# Set up the first plot for density estimation
theta_range = np.linspace(0, 1, 1000)
line1, = ax1.plot([], [], lw=2)
mean_line, = ax1.plot([], [], lw=2, linestyle='dashed', color='orange')  # Dashed line for mean
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 10)
ax1.set_title('Empirical Distribution Evolution')
ax1.set_xlabel('Theta (probability of heads)')
ax1.set_ylabel('Density')

# Set up the second plot for mean evolution
line2, = ax2.plot([], [], lw=2, color='orange')
ax2.set_xlim(0, len(coin_flips))
ax2.set_ylim(0, 1)
ax2.set_title('Evolution of Posterior Mean')
ax2.set_xlabel('Flip Number')
ax2.set_ylabel('Posterior Mean')

# Initialize shaded regions for both plots
std_shade1 = ax1.fill_between([], [], [], color='orange', alpha=0.2)
std_shade2 = ax2.fill_between([], [], [], color='orange', alpha=0.2)

# To store the evolution of the mean and std after each flip
posterior_means = []
posterior_stds = []

# Initialize KDE
kde_prior = KernelDensity(kernel='gaussian', bandwidth=0.05).fit(prior_theta[:, np.newaxis])

# Function to compute empirical likelihood based on current samples
def empirical_likelihood(theta,data):
    heads = np.sum(data==1)
    tails = np.sum(data==0)
    return theta**heads * (1-theta)**tails

# Function to initialize the plots
def init():
    line1.set_data([], [])
    mean_line.set_data([], [])
    return line1, mean_line, line2

# Function to update the plots after each coin flip
def update(frame):
    global prior_theta, std_shade1, std_shade2

    flip = coin_flips[frame]
    
    # Compute the likelihood of the new flip given the current theta (empirical prior)
    likelihoods = empirical_likelihood(prior_theta, np.array([flip]))

    # Weight the prior samples by the likelihood to get the posterior theta
    posterior_weights = likelihoods / np.sum(likelihoods)

    # Resample from the prior using the weights to generate posterior theta
    posterior_theta = np.random.choice(prior_theta, size=1000, p=posterior_weights)

    # Update the prior to be the posterior for the next step
    prior_theta = posterior_theta

    # Update the kernel density estimate for visulaization
    kde_prior = KernelDensity(kernel='gaussian', bandwidth=0.05).fit(posterior_theta[:, np.newaxis])
    log_density = kde_prior.score_samples(theta_range[:, np.newaxis])
    density = np.exp(log_density)

    # Update the line for the KDE (density plot)
    line1.set_data(theta_range, density)
   
    # Calculate the mean and standard deviation of the posterior theta
    posterior_mean = np.mean(posterior_theta)
    posterior_std = np.std(posterior_theta)
    posterior_means.append(posterior_mean)
    posterior_stds.append(posterior_std)
    
    # Update the line for the evolution of the mean
    line2.set_data(np.arange(len(posterior_means)), posterior_means)

    # Remove the previous shaded region (properly clearing it)
    for coll in ax1.collections:
        coll.remove()
    for coll in ax2.collections:
        coll.remove()

    # Update the shaded region for the Beta distribution plot (±1 std deviation)
    std_shade1 = ax1.fill_between(
        theta_range,
        density * (theta_range >= posterior_mean - posterior_std) * (theta_range <= posterior_mean + posterior_std),
        color='orange',
        alpha=0.4
    )

    # Update the shaded region for the mean plot (±1 std deviation)
    std_shade2 = ax2.fill_between(
        np.arange(len(posterior_means)),
        np.array(posterior_means) - np.array(posterior_stds),
        np.array(posterior_means) + np.array(posterior_stds),
        color='orange',
        alpha=0.4
    )
    
    # Update the vertical dashed line for the mean in the KDE plot
    mean_line.set_data([posterior_mean, posterior_mean], [0, 10]) 
    
    # Update titles
    ax1.set_title(f'After flip {frame+1}: {"Heads" if flip == 1 else "Tails"}')
    
    return line1, mean_line, line2

# Create the animation, including the initial prior frame
ani = FuncAnimation(fig, update, frames=len(coin_flips), init_func=init, blit=True, repeat=False)

# Display the animation in Jupyter Notebook using HTML
HTML(ani.to_jshtml())